# Figure 4a: slice medians by rejection type

Run cells in order using the **cardpy** Python environment. MD and FA use myocardial medians per slice. One gray box per rejection configuration pools both datasets. Dots and faint paired lines use one color per dataset (Dataset 1: PURL in blue; Dataset 2: CIMA in orange). HAP uses the same style.

The plotting code stays in `make_fig4.py`. Edit `DATA_ROOTS` below to select input datasets. Study IDs are prefixed with their dataset so slice pairing stays within the same participant and dataset. Outputs go into `figures/per_volunteer`.


In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display, Image

REPO = Path("/Users/ariel/Documents/GitHub/CarDpy")
DATA_ROOTS = {
    "PURL": Path("/Users/ariel/Downloads/rejection_comparison_purl"),
    "CIMA": Path("/Users/ariel/Downloads/rejection_comparison_cima 2"),
}
OUTPUT_DIR = REPO / "ml_rejection" / "figures" / "per_volunteer"

sys.path.insert(0, str(REPO / "ml_rejection"))
import make_fig4 as plot
for root in DATA_ROOTS.values():
    assert root.is_dir(), root


In [ ]:
# Load once; rerun this cell if input files change.
plot = importlib.reload(plot)
vox, hap, unc = plot.collect_datasets(DATA_ROOTS, slice_medians=True)
assert not vox.empty, f"No metric data under {DATA_ROOTS}"
cfgs = [c for c in plot.ORDER if c in set(vox.config)]
display(vox.groupby("dataset").agg(volunteers=("study", "nunique"), slice_median_rows=("value", "size")))
print("Volunteers:", sorted(vox.study.unique()))
print("Configurations:", cfgs)
print(f"Slice-median rows: {len(vox):,}; HAP rows: {len(hap)}")


In [ ]:
# Inspect each volunteer’s contribution; each black box pools all volunteers from both datasets.
# Counts are slice medians, not individual voxels.
summary = vox.groupby(["dataset", "study", "config", "metric"])["value"].agg(
    count="count", median="median", minimum="min", maximum="max"
)
display(summary)

# Source volunteer IDs; dots share a color within each dataset.
volunteers = sorted(vox.study.unique())
display(pd.DataFrame({"study": volunteers}))


In [ ]:
# Optional debugging: change these selectors to inspect one volunteer’s slice medians.
selected_study = volunteers[0]
selected_config = cfgs[0]
selected_metric = "MD"  # "MD" or "FA"
values = vox.loc[
    (vox.study == selected_study)
    & (vox.config == selected_config)
    & (vox.metric == selected_metric),
    "value",
]
display(values.describe())


In [ ]:
# Reload to pick up edits to make_fig4.py without restarting the kernel.
plot = importlib.reload(plot)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plot.FIGS = str(OUTPUT_DIR)
figure_path = plot.fig_a(vox, hap, cfgs)
summary.to_csv(OUTPUT_DIR / "fig4a_per_volunteer_values.csv")
print("Saved:", figure_path)
display(Image(filename=figure_path))
